# Day 7 lab: Memory, caching and checkpoints

**Goal:** Draw the executor memory layout from memory, compute each region, and match a symptom (spill, GC time, OOM, killed container) to its fix and to the right persistence tool.

**Time:** about 2 h 30 min. Checklist: 5.1–5.11, 6.10.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Chapter 4: Bug Busting — 30 min

Type the Spark UI section of the official guide (how a Python UDF shows up in the plan and the UI), then the UDF profilers: `spark.conf.set("spark.sql.pyspark.udf.profiler", "memory")`, run the UDF, `spark.profile.show(type="memory")`, then the same with `"perf"`. The memory profiler needs the `memory-profiler` package and real line numbers, so run it from a notebook or a .py file. Finish with the stack-trace settings. You get per-line memory use of your UDF: memory that lives in Python workers, outside the JVM heap.

[Chapter 4: Bug Busting](https://spark.apache.org/docs/latest/api/python/user_guide/bugbusting.html) · [Notebook source on GitHub](https://github.com/apache/spark/blob/v4.2.0/python/docs/source/user_guide/bugbusting.ipynb)

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 2. The persistence API — 10 min

Type the Examples of the official API pages for `DataFrame.persist`, `cache`, `unpersist`, `checkpoint` and `localCheckpoint`, printing `df.storageLevel` after each call.

[The persistence API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.persist.html) · [checkpoint](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.checkpoint.html) · [localCheckpoint](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.localCheckpoint.html)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 3. Exercise 1: cache the right thing — 30 min

Build `joined = orders.join(cust, "customer_id")` and run three different aggregations on it (by country, by segment, by status), timing the total: with no persistence, after `joined.cache()`, and after `joined.persist(StorageLevel.DISK_ONLY)`. Read the Storage tab (level, fraction cached, size in memory and on disk) and find `InMemoryTableScan` in the plans. `unpersist()` and watch the Storage tab empty. 

**Check:** about 2.6 s for the three reports without persistence; the first cached run is slower because it fills the cache (about 5 s), later runs take about 0.6 s; the Storage tab shows `Disk Memory Deserialized 1x Replicated`, 4 of 4 partitions, about 400 MiB in memory; DISK_ONLY holds about 290 MiB on disk and is slower than memory. Proves 5.11, 5.8.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day07.py)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 4. Exercise 2: cut a long lineage — 25 min

Build a DataFrame with a loop of 30 `withColumn` steps. `spark.sparkContext.setCheckpointDir("checkpoints")`, then compare `len(plan_text(df, "extended"))` before and after `df.checkpoint()` and `df.localCheckpoint()`: the plan shrinks to a `Scan ExistingRDD`. Look at the files written under `checkpoints/`. 

**Check:** the extended plan drops from about 17,000 characters to about 1,600. Proves 5.11, 6.10.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day07.py)

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 5. Compute the regions, then check the UI — 15 min

By hand, for 4, 10 and 16 GB heaps, compute reserved, usable, unified, protected storage and user memory (10 GB: 9,940 / 5,964 / 2,982 / 3,976 MB). Then open the Executors tab: `get_spark()` starts a 2 GB driver, and Storage Memory shows 1,048.8 MiB = (2048 − 300) × 0.6. The column says Storage, but it is the whole unified pool. Proves 5.1, 5.3, 5.10.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 6. Cause spill, then remove it — 20 min

Restart Python with `get_spark(driver_memory="1g")` so memory is tight. AQE off, `spark.sql.shuffle.partitions` = 2. `run(orders.orderBy("note"))` on `data/orders` and find Spill (Memory) and Spill (Disk) in the sort stage (about 300 MB and 190 MB). Set partitions to 200 and run it again: the spill disappears and the stage gets faster. Proves 5.2, 5.9.

In [ ]:
# Task 6: your code here


*Task 6 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 7. Driver out of memory — 20 min

Restart Python with `get_spark(conf={"spark.driver.maxResultSize": "64m"})` and `collect()` all of `data/orders`. Read the error. Fix it without raising the limit: aggregate first, `limit()`, or write the result to Parquet; then iterate with `toLocalIterator()` and note why it avoids the problem. Proves 5.5.

In [ ]:
# Task 7: your code here


*Task 7 notes: what you predicted, what the plan and the Spark UI showed.*